In [1]:
!pip install yfinance numpy pandas matplotlib scipy --quiet

In [2]:
# Install yfinance if not already installed

import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Define stock tickers and benchmark
tickers = ["0097.KL", "0215.KL", "1155.KL", "5347.KL", "5184.KL", "1295.KL", "5227.KL", "6742.KL", "2089.KL"]
benchmark_ticker = "^KLSE"  # FTSE Bursa Malaysia KLCI Index

# Download data
data = yf.download(tickers + [benchmark_ticker], start="2020-01-01", end="2024-12-31")['Close']
data.dropna(inplace=True)

# Separate benchmark and stock prices
benchmark = data[benchmark_ticker]
stocks = data[tickers]

# Daily returns
returns = stocks.pct_change().dropna()
benchmark_returns = benchmark.pct_change().dropna()
returns = returns.loc[benchmark_returns.index]

# Risk-free rate (annual)
rf = 0.03

# Annualized Return & Volatility
annual_return = returns.mean() * 252
annual_volatility = returns.std() * np.sqrt(252)

# Beta calculation: cov(stock, market) / var(market)
beta = {}
for stock in tickers:
    cov = np.cov(returns[stock], benchmark_returns)[0][1]
    var = np.var(benchmark_returns)
    beta[stock] = cov / var
beta = pd.Series(beta)

# Sharpe Ratio
sharpe_ratio = (annual_return - rf) / annual_volatility

# Treynor Ratio
treynor_ratio = (annual_return - rf) / beta

# Jensen's Alpha
market_return = benchmark_returns.mean() * 252
jensen_alpha = annual_return - (rf + beta * (market_return - rf))

# Max Drawdown
rolling_max = stocks.cummax()
drawdown = (stocks - rolling_max) / rolling_max
max_drawdown = drawdown.min()

# Historical VaR at 95% confidence
VaR_95 = returns.quantile(0.05)

# Combine all metrics
metrics = pd.DataFrame({
    'Annual Return': annual_return,
    'Annual Volatility': annual_volatility,
    'Sharpe Ratio': sharpe_ratio,
    'Treynor Ratio': treynor_ratio,
    'Jensen Alpha': jensen_alpha,
    'Max Drawdown': max_drawdown,
    'VaR (95%)': VaR_95,
    'Beta': beta
}).round(4)

# Display
print("📊 Performance Metrics (2020–2024):")
display(metrics)


YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  10 of 10 completed


📊 Performance Metrics (2020–2024):


,Annual Return,Annual Volatility,Sharpe Ratio,Treynor Ratio,Jensen Alpha,Max Drawdown,VaR (95%),Beta
0097.KL,0.1035,0.4085,0.1798,0.1198,0.0842,-0.7174,-0.0313,0.6130
0215.KL,0.3910,0.5379,0.6712,0.2406,0.3873,-0.7106,-0.0432,1.5003
1155.KL,0.1143,0.1539,0.5477,0.1055,0.0983,-0.1909,-0.0119,0.7991
5347.KL,0.0847,0.1982,0.2759,0.0626,0.0700,-0.3473,-0.0166,0.8741
5184.KL,0.0363,0.5562,0.0114,0.0056,0.0261,-0.8283,-0.0426,1.1298
1295.KL,0.0925,0.2288,0.2730,0.0541,0.0827,-0.3559,-0.0166,1.1543
5227.KL,0.0940,0.1927,0.3322,0.1608,0.0710,-0.2541,-0.0172,0.3981
6742.KL,0.4611,0.3605,1.1957,0.4452,0.4481,-0.4519,-0.0307,0.9683
2089.KL,0.2356,0.1562,1.3168,0.8156,0.2101,-0.1699,-0.0133,0.2521


In [5]:
import yfinance as yf
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import matplotlib.pyplot as plt

# Define tickers and benchmark
tickers = ["0097.KL", "0215.KL", "1155.KL", "5347.KL", "5184.KL", "1295.KL", "5227.KL", "6742.KL", "2089.KL"]
benchmark_ticker = "^KLSE"

# Download price data
data = yf.download(tickers + [benchmark_ticker], start="2020-01-01", end="2024-12-31")['Close'].dropna()
stocks = data[tickers]
benchmark = data[benchmark_ticker]

# Daily returns
returns = stocks.pct_change().dropna()
benchmark_returns = benchmark.pct_change().dropna()
returns = returns.loc[benchmark_returns.index]

# MVO to maximize return
mean_returns = returns.mean()
cov_matrix = returns.cov()
init_guess = [1 / len(tickers)] * len(tickers)
bounds = tuple((0, 1) for _ in tickers)
constraints = {'type': 'eq', 'fun': lambda x: np.sum(x) - 1}

def neg_return(weights, mean_returns):
    return -np.dot(weights, mean_returns)

opt = minimize(neg_return, init_guess, args=(mean_returns,), method='SLSQP', bounds=bounds, constraints=constraints)
weights = opt.x
weights_series = pd.Series(weights, index=tickers)

# Portfolio metrics
rf = 0.03
portfolio_return_annual = np.dot(weights, mean_returns) * 252
portfolio_volatility_annual = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights))) * np.sqrt(252)
portfolio_beta = sum(weights * [returns[stock].cov(benchmark_returns) for stock in tickers]) / benchmark_returns.var()
market_return_annual = benchmark_returns.mean() * 252
sharpe = (portfolio_return_annual - rf) / portfolio_volatility_annual
treynor = (portfolio_return_annual - rf) / portfolio_beta
jensen = portfolio_return_annual - (rf + portfolio_beta * (market_return_annual - rf))

# Drawdown
portfolio_growth = (returns @ weights + 1).cumprod()
rolling_max = portfolio_growth.cummax()
drawdown = (portfolio_growth - rolling_max) / rolling_max
max_drawdown = drawdown.min()

# Value at Risk
VaR_95 = np.percentile(returns @ weights, 5)

# Final dataframe
portfolio_metrics = pd.DataFrame({
    'Annual Return': [portfolio_return_annual],
    'Annual Volatility': [portfolio_volatility_annual],
    'Sharpe Ratio': [sharpe],
    'Treynor Ratio': [treynor],
    'Jensen Alpha': [jensen],
    'Max Drawdown': [max_drawdown],
    'VaR (95%)': [VaR_95],
    'Beta': [portfolio_beta]
}).round(4)

print("📊 Portfolio Performance Metrics (2020–2024):")
display(portfolio_metrics)

print("\n📈 Portfolio Weights:")
display(weights_series[weights_series > 0].round(4))


C:\Users\danial.zainudin\AppData\Local\Temp\ipykernel_22228\844225129.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers + [benchmark_ticker], start="2020-01-01", end="2024-12-31")['Close'].dropna()
[*********************100%***********************]  10 of 10 completed


📊 Portfolio Performance Metrics (2020–2024):


,Annual Return,Annual Volatility,Sharpe Ratio,Treynor Ratio,Jensen Alpha,Max Drawdown,VaR (95%),Beta
0,0.3987,0.2897,1.2725,0.3501,0.3872,-0.4082,-0.0247,1.053



📈 Portfolio Weights:


0097.KL    0.0000
0215.KL    0.3735
1155.KL    0.0000
5347.KL    0.0000
5184.KL    0.0000
1295.KL    0.0000
5227.KL    0.0000
6742.KL    0.4684
2089.KL    0.1581
dtype: float64